# 🏥 Health Institutions Database Overview

This notebook provides a clean overview of the health institutions data extracted from ISP/clinic images.

**Workflow:**
1. Load environment configuration
2. Process images from the `clients/` folder using Vision AI
3. Display extracted data and database statistics

## 1. Setup & Configuration

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from IPython.display import display, HTML, Image as IPImage

# Load environment variables from .env
load_dotenv()

# Verify API key is loaded
api_key = os.getenv('GOOGLE_AI_STUDIO')
print(f"✓ Google AI Studio API Key loaded: {'Yes' if api_key else 'No'}")
print(f"✓ Database URL: {os.getenv('DATABASE_URL', 'sqlite:///health_institutions.db')}")

ModuleNotFoundError: No module named 'dotenv'

## 2. Initialize Services

In [ ]:
# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our services
from health_scraper.services.vision_service import VisionExtractorService
from health_scraper.database.service import DatabaseService

# Initialize services
vision_service = VisionExtractorService()
db_service = DatabaseService()

print(f"✓ Vision Service initialized with model: {vision_service.model_name}")
print(f"✓ Database Service connected")

## 3. List Available Images

In [ ]:
# Get all images from the clients folder
clients_folder = project_root / 'clients'
image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.gif', '.bmp'}

images = [f for f in clients_folder.iterdir() 
          if f.is_file() and f.suffix.lower() in image_extensions]

print(f"📸 Found {len(images)} images in clients/ folder:\n")
for i, img in enumerate(images[:10], 1):
    print(f"  {i}. {img.name}")
if len(images) > 10:
    print(f"  ... and {len(images) - 10} more")

## 4. Process Images with Vision AI

This cell will extract health institution data from each image and save to the database.

In [ ]:
import asyncio
from tqdm.notebook import tqdm

# Process images
results = []

async def process_all_images():
    for img_path in tqdm(images, desc="Processing images"):
        try:
            result = await vision_service.extract_from_image(
                img_path, 
                save_to_database=True
            )
            results.append({
                'filename': img_path.name,
                'success': result.success,
                'is_health_related': result.is_health_related,
                'confidence': result.confidence,
                'is_duplicate': result.is_duplicate,
                'institution_id': result.existing_id,
                'name': result.extracted_data.get('name') if result.extracted_data else None,
                'message': result.message
            })
        except Exception as e:
            results.append({
                'filename': img_path.name,
                'success': False,
                'message': str(e)
            })

# Run the async function
await process_all_images()

# Show results summary
df_results = pd.DataFrame(results)
print("\n📊 Processing Summary:")
print(f"  • Total images: {len(results)}")
print(f"  • Successful: {df_results['success'].sum()}")
print(f"  • Health-related: {df_results['is_health_related'].sum() if 'is_health_related' in df_results else 0}")
print(f"  • New institutions: {(~df_results['is_duplicate'].fillna(True)).sum() if 'is_duplicate' in df_results else 0}")
print(f"  • Duplicates: {df_results['is_duplicate'].sum() if 'is_duplicate' in df_results else 0}")

## 5. Extraction Results

In [ ]:
# Display extraction results in a clean table
display_cols = ['filename', 'is_health_related', 'confidence', 'name', 'is_duplicate', 'institution_id']
available_cols = [c for c in display_cols if c in df_results.columns]

df_display = df_results[available_cols].copy()
df_display['confidence'] = df_display['confidence'].apply(lambda x: f"{x:.0%}" if pd.notna(x) else '-')

# Style the dataframe
def highlight_health(row):
    if row.get('is_health_related') == True:
        return ['background-color: #d4edda'] * len(row)
    return [''] * len(row)

styled = df_display.style.apply(highlight_health, axis=1)
display(styled)

## 6. Database Overview

In [ ]:
# Get database statistics
from health_scraper.database.models import InstitutionDB, ContactDB, SocialMediaDB

session = db_service.get_session()

# Count records
total_institutions = session.query(InstitutionDB).count()
total_contacts = session.query(ContactDB).count()
total_social = session.query(SocialMediaDB).count()

print("📊 Database Statistics")
print("=" * 40)
print(f"  🏥 Total Institutions: {total_institutions}")
print(f"  👤 Total Contacts: {total_contacts}")
print(f"  📱 Social Media Profiles: {total_social}")

session.close()

In [ ]:
# Load all institutions into a DataFrame
session = db_service.get_session()
institutions = session.query(InstitutionDB).all()

# Convert to DataFrame
data = []
for inst in institutions:
    data.append({
        'ID': inst.id,
        'Nombre': inst.name,
        'Tipo': inst.institution_type,
        'NIT': inst.nit,
        'Ciudad': inst.city,
        'Departamento': inst.department,
        'Dirección': inst.address,
        'Teléfono': inst.phone,
        'Email': inst.email,
        'Website': inst.website,
        'Servicios': ', '.join(inst.services) if inst.services else '',
        'Fecha Registro': inst.scraped_at.strftime('%Y-%m-%d %H:%M') if inst.scraped_at else ''
    })

df_institutions = pd.DataFrame(data)
session.close()

print(f"\n🏥 Health Institutions in Database ({len(df_institutions)} records)\n")
display(df_institutions)

## 7. Data Distribution Analysis

In [ ]:
if len(df_institutions) > 0:
    print("📍 Distribution by City:")
    city_counts = df_institutions['Ciudad'].value_counts()
    for city, count in city_counts.items():
        if pd.notna(city) and city:
            print(f"  • {city}: {count}")
    
    print("\n🏢 Distribution by Type:")
    type_counts = df_institutions['Tipo'].value_counts()
    for tipo, count in type_counts.items():
        if pd.notna(tipo) and tipo:
            print(f"  • {tipo}: {count}")
else:
    print("No institutions in database yet. Run the image processing cell first.")

## 8. Export Data

In [ ]:
# Export to CSV and Excel
if len(df_institutions) > 0:
    # CSV export
    csv_path = project_root / 'exports' / 'institutions_export.csv'
    csv_path.parent.mkdir(exist_ok=True)
    df_institutions.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"✓ CSV exported to: {csv_path}")
    
    # Excel export
    excel_path = project_root / 'exports' / 'institutions_export.xlsx'
    df_institutions.to_excel(excel_path, index=False, sheet_name='Instituciones')
    print(f"✓ Excel exported to: {excel_path}")
else:
    print("No data to export.")